In [3]:
# ============================================================
# E2 — Confidence-weighted SFT (CW-SFT) — ALL 6 QWEN2.5 MODELS
# Loss:  L_i = alpha_i * L_SFT,i
# alpha_i from Gemini top-K margin (mean margin -> sigmoid -> mean-normalized + clipped)
# - Prompt tokens masked (labels=-100)
# - Uses ONLY frozen train set: clinical_pharm_teacher_train_6000.jsonl
# - QLoRA (4-bit) + LoRA adapters
# ============================================================

!pip -q install -U transformers peft accelerate bitsandbytes sentencepiece

import os, json, math, re
from typing import List, Dict, Any, Tuple

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model
from google.colab import drive

drive.mount("/content/drive")
# ----------------------------
# CONFIG
# ----------------------------
BASE_DIR = "/content/drive/MyDrive/DL_Local"
PROMPTS_PATH = os.path.join(BASE_DIR, "clinical_pharm_prompts_10000.jsonl")
TEACHER_6K_PATH = os.path.join(BASE_DIR, "clinical_pharm_teacher_train_6000.jsonl")

OUT_DIR = os.path.join(BASE_DIR, "cw_sft_runs")  # ✅ E2 outputs here
os.makedirs(OUT_DIR, exist_ok=True)

MAX_SEQ_LEN = 2048
EPOCHS = 2
LR = 2e-4
BATCH_SIZE = 1
GRAD_ACC = 32
SEED = 42

# alpha clipping
ALPHA_MIN = 0.25
ALPHA_MAX = 3.0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

STUDENTS = {
    "qwen25_1p5b_base":     {"path": "Qwen/Qwen2.5-1.5B",          "is_instruct": False},
    "qwen25_1p5b_instruct": {"path": "Qwen/Qwen2.5-1.5B-Instruct", "is_instruct": True},
    "qwen25_3b_base":       {"path": "Qwen/Qwen2.5-3B",            "is_instruct": False},
    "qwen25_3b_instruct":   {"path": "Qwen/Qwen2.5-3B-Instruct",   "is_instruct": True},
    "qwen25_7b_base":       {"path": "Qwen/Qwen2.5-7B",            "is_instruct": False},
    "qwen25_7b_instruct":   {"path": "Qwen/Qwen2.5-7B-Instruct",   "is_instruct": True},
}

# ----------------------------
# IO helpers
# ----------------------------
def load_jsonl(path: str) -> List[Dict[str, Any]]:
    out = []
    with open(path, "r", encoding="utf-8") as f:
        for l in f:
            l = l.strip()
            if not l:
                continue
            out.append(json.loads(l))
    return out

prompts = {r["id"]: r["prompt"] for r in load_jsonl(PROMPTS_PATH)}

teacher_rows = [
    r for r in load_jsonl(TEACHER_6K_PATH)
    if (r.get("status") == "ok" and r.get("teacher_text") and r.get("split") == "train")
]
print(f"Loaded {len(teacher_rows)} teacher samples (FROZEN TRAIN SET): {TEACHER_6K_PATH}")

# ----------------------------
# Confidence: top-K margin -> mean -> sigmoid -> mean-normalized alpha
# ----------------------------
def _sigmoid(x: float) -> float:
    # stable sigmoid for moderate x
    if x >= 0:
        z = math.exp(-x)
        return 1.0 / (1.0 + z)
    else:
        z = math.exp(x)
        return z / (1.0 + z)

def compute_mean_margin(logprobs_steps: List[Dict[str, Any]]) -> float | None:
    """
    m(t)=logp(chosen)-max(logp(alt)) excluding chosen token string.
    Returns mean margin across valid steps, or None if no valid steps.
    """
    if not logprobs_steps:
        return None

    margins = []
    for step in logprobs_steps:
        ch = step.get("chosen", {})
        topk = step.get("topk", [])
        ch_lp = ch.get("logprob", None)
        ch_tok = ch.get("token", None)
        if ch_lp is None or not topk:
            continue

        alt_lps = []
        for tc in topk:
            # exclude chosen token (string match)
            if ch_tok is not None and tc.get("token", None) == ch_tok:
                continue
            lp = tc.get("logprob", None)
            if lp is not None:
                alt_lps.append(lp)

        if not alt_lps:
            continue

        margins.append(float(ch_lp) - float(max(alt_lps)))

    if not margins:
        return None
    return float(sum(margins) / len(margins))

def compute_confidence(row: Dict[str, Any]) -> float:
    mm = compute_mean_margin(row.get("logprobs_steps", None) or [])
    if mm is None:
        return float("nan")
    return _sigmoid(mm)

# Compute mean confidence E[c] over available rows (skip NaNs)
confs = []
for r in teacher_rows:
    c = compute_confidence(r)
    if not (c != c):  # not NaN
        confs.append(c)

MEAN_C = float(sum(confs) / max(1, len(confs)))
print(f"Mean confidence E[c] over train rows (valid only): {MEAN_C:.6f} (n_valid={len(confs)})")

def compute_alpha(row: Dict[str, Any]) -> float:
    c = compute_confidence(row)
    if (c != c) or MEAN_C <= 0:   # NaN or degenerate
        return 1.0
    a = c / MEAN_C
    if a < ALPHA_MIN:
        a = ALPHA_MIN
    elif a > ALPHA_MAX:
        a = ALPHA_MAX
    return float(a)

# ----------------------------
# Dataset (uniform token loss, but returns alpha_i)
# ----------------------------
class CWSFTDataset(Dataset):
    def __init__(self, rows, tokenizer, is_instruct):
        self.rows = rows
        self.tokenizer = tokenizer
        self.is_instruct = is_instruct

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        ex_id = r["id"]
        prompt = prompts[ex_id]
        answer = r["teacher_text"]

        if self.is_instruct and hasattr(self.tokenizer, "apply_chat_template"):
            prompt_text = self.tokenizer.apply_chat_template(
                [{"role": "user", "content": prompt}],
                tokenize=False,
                add_generation_prompt=True
            )
        else:
            prompt_text = prompt

        full_text = prompt_text + answer
        tok = self.tokenizer(
            full_text,
            truncation=True,
            max_length=MAX_SEQ_LEN,
            return_offsets_mapping=True
        )

        input_ids = tok["input_ids"]
        offsets = tok["offset_mapping"]

        labels = [-100] * len(input_ids)
        answer_start = len(prompt_text)
        for i, (s, e) in enumerate(offsets):
            if e <= s:
                continue
            if s >= answer_start:
                labels[i] = input_ids[i]

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "attention_mask": torch.ones(len(input_ids), dtype=torch.long),
            "alpha": torch.tensor(compute_alpha(r), dtype=torch.float),
        }

# ----------------------------
# Collator (pads labels; alpha is [B])
# ----------------------------
class CWSFTCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        batch = {
            "input_ids": [f["input_ids"] for f in features],
            "attention_mask": [f["attention_mask"] for f in features],
            "labels": [f["labels"] for f in features],
            "alpha": torch.stack([f["alpha"] for f in features], dim=0),
        }

        padded = self.tokenizer.pad(
            {"input_ids": batch["input_ids"], "attention_mask": batch["attention_mask"]},
            padding=True,
            return_tensors="pt"
        )
        max_len = padded["input_ids"].shape[1]

        labels = torch.full((len(features), max_len), -100, dtype=torch.long)
        for i, l in enumerate(batch["labels"]):
            labels[i, : l.shape[0]] = l

        padded["labels"] = labels
        padded["alpha"] = batch["alpha"]
        return padded

# ----------------------------
# Trainer (alpha-weighted per-sample loss)
# ----------------------------
class AlphaWeightedSFTTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        alpha = inputs.pop("alpha")  # [B]
        outputs = model(**inputs)
        logits = outputs.logits  # [B, T, V]

        # Shift for causal LM
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()

        vocab = shift_logits.size(-1)
        loss_fct = torch.nn.CrossEntropyLoss(reduction="none")

        token_loss = loss_fct(
            shift_logits.view(-1, vocab),
            shift_labels.view(-1)
        ).view(shift_labels.size())  # [B, T-1]

        active = (shift_labels != -100).float()  # [B, T-1]
        denom = active.sum(dim=1).clamp(min=1.0)  # [B]

        per_sample_loss = (token_loss * active).sum(dim=1) / denom  # [B]
        loss = (per_sample_loss * alpha).mean()

        return (loss, outputs) if return_outputs else loss

# ----------------------------
# TRAINING LOOP
# ----------------------------
for model_name, cfg in STUDENTS.items():
    print(f"\n===== E2 CW-SFT TRAINING {model_name} =====")

    out_path = os.path.join(OUT_DIR, model_name)
    os.makedirs(out_path, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(cfg["path"], trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        cfg["path"],
        load_in_4bit=True,
        device_map="auto",
        trust_remote_code=True,
    )

    lora_cfg = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_cfg)
    model.train()

    dataset = CWSFTDataset(teacher_rows, tokenizer, cfg["is_instruct"])
    collator = CWSFTCollator(tokenizer)

    args = TrainingArguments(
    output_dir=out_path,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR,
    logging_steps=50,
    save_strategy="no",
    fp16=True,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,  # ✅ IMPORTANT: keep "alpha"
)


    trainer = AlphaWeightedSFTTrainer(
        model=model,
        args=args,
        train_dataset=dataset,
        data_collator=collator,
    )

    trainer.train()
    model.save_pretrained(out_path)
    tokenizer.save_pretrained(out_path)
    print(f"✅ Finished E2 CW-SFT {model_name}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 5000 teacher samples (FROZEN TRAIN SET): /content/drive/MyDrive/DL_Local/clinical_pharm_teacher_train_6000.jsonl
Mean confidence E[c] over train rows (valid only): 0.999763 (n_valid=5000)

===== E2 CW-SFT TRAINING qwen25_1p5b_base =====


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss
50,43.007100
100,36.399500
150,33.979900
200,31.780100
250,31.450900
300,31.161100


✅ Finished E2 CW-SFT qwen25_1p5b_base

===== E2 CW-SFT TRAINING qwen25_1p5b_instruct =====


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss
50,34.852300
100,28.931100
150,26.960600
200,24.839400
250,24.472800
300,24.208000


✅ Finished E2 CW-SFT qwen25_7b_instruct


In [4]:
# ============================================================
# E3 — Confidence-weighted WSFT (CW-WSFT) — ALL 6 QWEN2.5 MODELS
# Loss:  L_i = alpha_i * L_WSFT,i
# WSFT weights: format=1, decision=2, explanation=2
# WSFT token weights normalized per-sample to mean=1 (on supervised tokens)
# alpha_i mean-normalized + clipped (same definition as E2)
# - Prompt tokens masked (labels=-100)
# - Uses ONLY frozen train set: clinical_pharm_teacher_train_6000.jsonl
# - QLoRA (4-bit) + LoRA adapters
# ============================================================

!pip -q install -U transformers peft accelerate bitsandbytes sentencepiece

import os, json, math, re
from typing import List, Dict, Any, Tuple

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model

# ----------------------------
# CONFIG (same as E2, but separate OUT_DIR)
# ----------------------------
BASE_DIR = "/content/drive/MyDrive/DL_Local"
PROMPTS_PATH = os.path.join(BASE_DIR, "clinical_pharm_prompts_10000.jsonl")
TEACHER_6K_PATH = os.path.join(BASE_DIR, "clinical_pharm_teacher_train_6000.jsonl")

OUT_DIR = os.path.join(BASE_DIR, "cw_wsft_runs")  # ✅ E3 outputs here
os.makedirs(OUT_DIR, exist_ok=True)

MAX_SEQ_LEN = 2048
EPOCHS = 2
LR = 2e-4
BATCH_SIZE = 1
GRAD_ACC = 32
SEED = 42

# WSFT weights
W_FORMAT = 1.0
W_DECISION = 2.0
W_EXPL = 2.0
NORMALIZE_WSFT_WEIGHTS_PER_SAMPLE = True  # mean weight over supervised tokens = 1

# alpha clipping
ALPHA_MIN = 0.25
ALPHA_MAX = 3.0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

STUDENTS = {
    "qwen25_1p5b_base":     {"path": "Qwen/Qwen2.5-1.5B",          "is_instruct": False},
    "qwen25_1p5b_instruct": {"path": "Qwen/Qwen2.5-1.5B-Instruct", "is_instruct": True},
    "qwen25_3b_base":       {"path": "Qwen/Qwen2.5-3B",            "is_instruct": False},
    "qwen25_3b_instruct":   {"path": "Qwen/Qwen2.5-3B-Instruct",   "is_instruct": True},
    "qwen25_7b_base":       {"path": "Qwen/Qwen2.5-7B",            "is_instruct": False},
    "qwen25_7b_instruct":   {"path": "Qwen/Qwen2.5-7B-Instruct",   "is_instruct": True},
}

# ----------------------------
# IO helpers
# ----------------------------
def load_jsonl(path: str) -> List[Dict[str, Any]]:
    out = []
    with open(path, "r", encoding="utf-8") as f:
        for l in f:
            l = l.strip()
            if not l:
                continue
            out.append(json.loads(l))
    return out

prompts = {r["id"]: r["prompt"] for r in load_jsonl(PROMPTS_PATH)}

teacher_rows = [
    r for r in load_jsonl(TEACHER_6K_PATH)
    if (r.get("status") == "ok" and r.get("teacher_text") and r.get("split") == "train")
]
print(f"Loaded {len(teacher_rows)} teacher samples (FROZEN TRAIN SET): {TEACHER_6K_PATH}")

# ----------------------------
# Confidence (same as E2)
# ----------------------------
def _sigmoid(x: float) -> float:
    if x >= 0:
        z = math.exp(-x)
        return 1.0 / (1.0 + z)
    else:
        z = math.exp(x)
        return z / (1.0 + z)

def compute_mean_margin(logprobs_steps: List[Dict[str, Any]]) -> float | None:
    if not logprobs_steps:
        return None
    margins = []
    for step in logprobs_steps:
        ch = step.get("chosen", {})
        topk = step.get("topk", [])
        ch_lp = ch.get("logprob", None)
        ch_tok = ch.get("token", None)
        if ch_lp is None or not topk:
            continue
        alt_lps = []
        for tc in topk:
            if ch_tok is not None and tc.get("token", None) == ch_tok:
                continue
            lp = tc.get("logprob", None)
            if lp is not None:
                alt_lps.append(lp)
        if not alt_lps:
            continue
        margins.append(float(ch_lp) - float(max(alt_lps)))
    if not margins:
        return None
    return float(sum(margins) / len(margins))

def compute_confidence(row: Dict[str, Any]) -> float:
    mm = compute_mean_margin(row.get("logprobs_steps", None) or [])
    if mm is None:
        return float("nan")
    return _sigmoid(mm)

confs = []
for r in teacher_rows:
    c = compute_confidence(r)
    if not (c != c):
        confs.append(c)
MEAN_C = float(sum(confs) / max(1, len(confs)))
print(f"Mean confidence E[c] over train rows (valid only): {MEAN_C:.6f} (n_valid={len(confs)})")

def compute_alpha(row: Dict[str, Any]) -> float:
    c = compute_confidence(row)
    if (c != c) or MEAN_C <= 0:
        return 1.0
    a = c / MEAN_C
    if a < ALPHA_MIN:
        a = ALPHA_MIN
    elif a > ALPHA_MAX:
        a = ALPHA_MAX
    return float(a)

# ----------------------------
# Section span finder (same heuristic as E1)
# ----------------------------
_DECISION_PATTERNS = [
    r'"decision"\s*:\s*', r"'decision'\s*:\s*", r"\bDecision\s*:\s*", r"\bDECISION\s*:\s*"
]
_EXPL_PATTERNS = [
    r'"explanation"\s*:\s*', r"'explanation'\s*:\s*", r"\bExplanation\s*:\s*", r"\bEXPLANATION\s*:\s*",
    r'"reasoning"\s*:\s*', r"'reasoning'\s*:\s*", r"\bReasoning\s*:\s*", r"\bREASONING\s*:\s*"
]

def _find_span_by_markers(text: str, start_markers: List[str], end_markers: List[str]) -> Tuple[int, int]:
    start_end = -1
    for pat in start_markers:
        m = re.search(pat, text)
        if m:
            start_end = m.end()
            break
    if start_end == -1:
        return (-1, -1)

    end_pos = len(text)
    for pat in end_markers:
        m2 = re.search(pat, text[start_end:])
        if m2:
            end_pos = min(end_pos, start_end + m2.start())
    return (start_end, end_pos)

def get_section_spans(answer_text: str) -> Dict[str, List[Tuple[int, int]]]:
    spans = {"decision": [], "explanation": []}
    d_span = _find_span_by_markers(answer_text, _DECISION_PATTERNS, _EXPL_PATTERNS)
    if d_span != (-1, -1) and d_span[0] < d_span[1]:
        spans["decision"].append(d_span)

    e_span = _find_span_by_markers(answer_text, _EXPL_PATTERNS, _DECISION_PATTERNS)
    if e_span != (-1, -1) and e_span[0] < e_span[1]:
        spans["explanation"].append(e_span)

    return spans

# ----------------------------
# Dataset (returns token weights + alpha)
# ----------------------------
class CWWSFTDataset(Dataset):
    def __init__(self, rows, tokenizer, is_instruct):
        self.rows = rows
        self.tokenizer = tokenizer
        self.is_instruct = is_instruct

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        ex_id = r["id"]
        prompt = prompts[ex_id]
        answer = r["teacher_text"]

        if self.is_instruct and hasattr(self.tokenizer, "apply_chat_template"):
            prompt_text = self.tokenizer.apply_chat_template(
                [{"role": "user", "content": prompt}],
                tokenize=False,
                add_generation_prompt=True
            )
        else:
            prompt_text = prompt

        full_text = prompt_text + answer

        tok = self.tokenizer(
            full_text,
            truncation=True,
            max_length=MAX_SEQ_LEN,
            return_offsets_mapping=True
        )
        input_ids = tok["input_ids"]
        offsets = tok["offset_mapping"]

        labels = [-100] * len(input_ids)
        weights = [0.0] * len(input_ids)

        answer_start_abs = len(prompt_text)

        spans = get_section_spans(answer)
        decision_spans_abs = [(answer_start_abs + s, answer_start_abs + e) for (s, e) in spans["decision"]]
        expl_spans_abs     = [(answer_start_abs + s, answer_start_abs + e) for (s, e) in spans["explanation"]]

        def in_any_span(ts: int, te: int, span_list: List[Tuple[int, int]]) -> bool:
            for (s, e) in span_list:
                if te > s and ts < e:
                    return True
            return False

        for i, (s, e) in enumerate(offsets):
            if e <= s:
                continue
            if s >= answer_start_abs:
                labels[i] = input_ids[i]
                w = W_FORMAT
                if in_any_span(s, e, decision_spans_abs):
                    w = W_DECISION
                if in_any_span(s, e, expl_spans_abs):
                    w = W_EXPL
                weights[i] = float(w)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "attention_mask": torch.ones(len(input_ids), dtype=torch.long),
            "weights": torch.tensor(weights, dtype=torch.float),
            "alpha": torch.tensor(compute_alpha(r), dtype=torch.float),
        }

# ----------------------------
# Collator (pads labels + weights; alpha is [B])
# ----------------------------
class CWWSFTCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        batch = {
            "input_ids": [f["input_ids"] for f in features],
            "attention_mask": [f["attention_mask"] for f in features],
            "labels": [f["labels"] for f in features],
            "weights": [f["weights"] for f in features],
            "alpha": torch.stack([f["alpha"] for f in features], dim=0),
        }

        padded = self.tokenizer.pad(
            {"input_ids": batch["input_ids"], "attention_mask": batch["attention_mask"]},
            padding=True,
            return_tensors="pt"
        )
        max_len = padded["input_ids"].shape[1]

        labels = torch.full((len(features), max_len), -100, dtype=torch.long)
        weights = torch.zeros((len(features), max_len), dtype=torch.float)

        for i, (l, w) in enumerate(zip(batch["labels"], batch["weights"])):
            labels[i, : l.shape[0]] = l
            weights[i, : w.shape[0]] = w

        padded["labels"] = labels
        padded["weights"] = weights
        padded["alpha"] = batch["alpha"]
        return padded

# ----------------------------
# Trainer (WSFT token weights + alpha_i)
# ----------------------------
class AlphaWeightedWSFTTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")     # [B, T]
        weights = inputs.pop("weights")   # [B, T]
        alpha = inputs.pop("alpha")       # [B]
        outputs = model(**inputs)
        logits = outputs.logits           # [B, T, V]

        # Shift
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        shift_weights = weights[:, 1:].contiguous()

        vocab = shift_logits.size(-1)
        loss_fct = torch.nn.CrossEntropyLoss(reduction="none")

        token_loss = loss_fct(
            shift_logits.view(-1, vocab),
            shift_labels.view(-1)
        ).view(shift_labels.size())  # [B, T-1]

        active = (shift_labels != -100).float()  # [B, T-1]
        w = shift_weights * active               # [B, T-1]

        if NORMALIZE_WSFT_WEIGHTS_PER_SAMPLE:
            denom = active.sum(dim=1, keepdim=True).clamp(min=1.0)
            mean_w = (w.sum(dim=1, keepdim=True) / denom).clamp(min=1e-6)
            w = w / mean_w

        denom_tok = active.sum(dim=1).clamp(min=1.0)  # [B]
        per_sample_loss = (token_loss * w).sum(dim=1) / denom_tok  # [B]

        loss = (per_sample_loss * alpha).mean()
        return (loss, outputs) if return_outputs else loss

# ----------------------------
# TRAINING LOOP
# ----------------------------
for model_name, cfg in STUDENTS.items():
    print(f"\n===== E3 CW-WSFT TRAINING {model_name} =====")

    out_path = os.path.join(OUT_DIR, model_name)
    os.makedirs(out_path, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(cfg["path"], trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        cfg["path"],
        load_in_4bit=True,
        device_map="auto",
        trust_remote_code=True,
    )

    lora_cfg = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_cfg)
    model.train()

    dataset = CWWSFTDataset(teacher_rows, tokenizer, cfg["is_instruct"])
    collator = CWWSFTCollator(tokenizer)

    args = TrainingArguments(
    output_dir=out_path,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR,
    logging_steps=50,
    save_strategy="no",
    fp16=True,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,  # ✅ keep "alpha" and "weights"
)


    trainer = AlphaWeightedWSFTTrainer(
        model=model,
        args=args,
        train_dataset=dataset,
        data_collator=collator,
    )

    trainer.train()
    model.save_pretrained(out_path)
    tokenizer.save_pretrained(out_path)
    print(f"✅ Finished E3 CW-WSFT {model_name}")


Loaded 5000 teacher samples (FROZEN TRAIN SET): /content/drive/MyDrive/DL_Local/clinical_pharm_teacher_train_6000.jsonl
Mean confidence E[c] over train rows (valid only): 0.999763 (n_valid=5000)

===== E3 CW-WSFT TRAINING qwen25_1p5b_base =====


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss
50,43.297300
100,36.704200
150,34.283300
200,32.079200
250,31.740900
300,31.459000


✅ Finished E3 CW-WSFT qwen25_1p5b_base

===== E3 CW-WSFT TRAINING qwen25_1p5b_instruct =====


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


...

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss
50,35.195100
100,29.212200
150,27.221500
200,25.081100
250,24.708000
300,24.442400


✅ Finished E3 CW-WSFT qwen25_7b_instruct


In [5]:
# ============================================================
# COLAB — Unassign runtime (official API)
# ============================================================

from google.colab import runtime

print("✅ Job finished. Unassigning runtime...")
runtime.unassign()


✅ Job finished. Unassigning runtime...
